In [ ]:
# Text classification does not need torchvision; mismatched torchvision builds can break transformers imports.
%pip uninstall -y torchvision torchaudio torchtext
%pip install -q -U "transformers>=4.48,<5" accelerate scikit-learn pandas seaborn matplotlib nltk


In [ ]:
# --- Shared setup: ELECTRA + RoBERTa hybrid experiments ---
import copy
import os
import platform
import random
import re
import time
import sys
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import transformers
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords, wordnet
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

os.environ["WANDB_DISABLED"] = "true"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

SPLIT_SEED = 42
TRAIN_SEEDS = [42, 123, 2024]
RANDOM_STATE = SPLIT_SEED
ELECTRA_MODEL_NAME = "google/electra-base-discriminator"
ROBERTA_MODEL_NAME = "roberta-base"
DATA_URL = "https://raw.githubusercontent.com/egbertbritannia-cpu/rbl-requirements-classification/main/data/exp/promise_exp.csv"
LOCAL_DATA_PATHS = [
    Path("data/exp/promise_exp.csv"),
    Path("../data/exp/promise_exp.csv"),
]

MAX_LENGTH = 128
NUM_EPOCHS = 30
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
EARLY_STOPPING_PATIENCE = None  # Disabled so SERF experiments always run all 30 epochs.
USE_GRADIENT_CHECKPOINTING = True
BENCHMARK_WARMUP_ROUNDS = 2
BENCHMARK_REPEATS = 10

AUGMENTATION_TARGETS = {
    "FT": 200,
    "L": 200,
    "MN": 200,
    "A": 200,
    "LF": 200,
    "SC": 200,
    "PO": 200,
    "PE": 200,
    "O": 200,
    "US": 200,
    "SE": 100,
}
KEEP_WORDS = {"not", "no", "shall", "must", "should", "will", "cannot", "never", "only"}

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SPLIT_SEED)

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
stop_words = set(stopwords.words("english")) - KEEP_WORDS


def normalize_requirement_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def get_synonyms(word):
    clean_word = re.sub(r"[^a-zA-Z]", "", str(word).lower())
    if not clean_word:
        return []
    synonyms = set()
    for syn in wordnet.synsets(clean_word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ").lower()
            if synonym != clean_word:
                synonyms.add(synonym)
    return list(synonyms)


def synonym_replacement(text, n=1):
    words = str(text).split()
    new_words = words.copy()
    candidates = list({
        word for word in words
        if re.sub(r"[^a-zA-Z]", "", word).lower() not in stop_words and len(word) > 3
    })
    random.shuffle(candidates)

    replaced = 0
    for word in candidates:
        synonyms = get_synonyms(word)
        if synonyms:
            synonym = random.choice(synonyms)
            new_words = [synonym if current == word else current for current in new_words]
            replaced += 1
        if replaced >= n:
            break
    return " ".join(new_words)


def phrase_insertion(text, n=1):
    phrases = ["the system shall", "must be able to", "should ensure that"]
    words = str(text).split()
    if not words:
        return str(text)
    for _ in range(n):
        words.insert(random.randint(0, len(words) - 1), random.choice(phrases))
    return " ".join(words)


def random_swap(text, n=1):
    words = str(text).split()
    if len(words) < 2:
        return str(text)
    for _ in range(n):
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return " ".join(words)


def random_deletion(text, p=0.1):
    words = str(text).split()
    if len(words) <= 1:
        return str(text)
    remaining = [word for word in words if random.random() > p]
    return " ".join(remaining) if remaining else random.choice(words)


def augment_requirement_text(text):
    text = synonym_replacement(text, n=1)
    text = phrase_insertion(text, n=1)
    text = random_swap(text, n=1)
    text = random_deletion(text, p=0.1)
    return normalize_requirement_text(text)


def load_promise_exp():
    for path in LOCAL_DATA_PATHS:
        if path.exists():
            print(f"Loaded PROMISE expanded dataset from local path: {path}")
            return pd.read_csv(path)
    print(f"Loaded PROMISE expanded dataset from URL: {DATA_URL}")
    return pd.read_csv(DATA_URL)


df = load_promise_exp()
df = df[["RequirementText", "class"]].dropna().reset_index(drop=True)
df["RequirementText"] = df["RequirementText"].astype(str).map(normalize_requirement_text)
df["class"] = df["class"].astype(str).str.strip().str.upper()
df = df[df["RequirementText"] != ""].reset_index(drop=True)
df = df[df["class"].map(df["class"].value_counts()) > 1].reset_index(drop=True)
df["model_text"] = df["RequirementText"]

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["class"])
id2label = {idx: label for idx, label in enumerate(label_encoder.classes_)}
label2id = {label: idx for idx, label in id2label.items()}
num_labels = len(label_encoder.classes_)

train_df, val_test_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label"],
    random_state=SPLIT_SEED,
)
val_df, test_df = train_test_split(
    val_test_df,
    test_size=0.5,
    stratify=val_test_df["label"],
    random_state=SPLIT_SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Full distribution:", dict(df["class"].value_counts()))
print("Train distribution:", dict(train_df["class"].value_counts()))
print("Validation distribution:", dict(val_df["class"].value_counts()))
print("Test distribution:", dict(test_df["class"].value_counts()))
print("Labels:", list(label_encoder.classes_))
print("Using raw requirement text; no stopword/punctuation/digit removal.")

In [ ]:
# --- Dataset, metrics, and PyTorch training loop ---
electra_tokenizer = AutoTokenizer.from_pretrained(ELECTRA_MODEL_NAME, use_fast=True)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME, use_fast=True)


class ElectraRobertaDataset(Dataset):
    def __init__(self, frame):
        texts = frame["model_text"].astype(str).tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.electra = electra_tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_token_type_ids=True,
        )
        self.roberta = roberta_tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            "electra_input_ids": torch.tensor(self.electra["input_ids"][idx], dtype=torch.long),
            "electra_attention_mask": torch.tensor(self.electra["attention_mask"][idx], dtype=torch.long),
            "roberta_input_ids": torch.tensor(self.roberta["input_ids"][idx], dtype=torch.long),
            "roberta_attention_mask": torch.tensor(self.roberta["attention_mask"][idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }
        if "token_type_ids" in self.electra:
            item["electra_token_type_ids"] = torch.tensor(self.electra["token_type_ids"][idx], dtype=torch.long)
        return item


def make_loaders(train_frame, seed=None):
    train_ds = ElectraRobertaDataset(train_frame)
    val_ds = ElectraRobertaDataset(val_df)
    test_ds = ElectraRobertaDataset(test_df)
    generator = None
    if seed is not None:
        generator = torch.Generator()
        generator.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, generator=generator)
    val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE)
    return train_loader, val_loader, test_loader


def compute_class_weights(frame):
    counts = frame["label"].value_counts().sort_index()
    weights = len(frame) / (len(counts) * counts)
    return torch.tensor(weights.values, dtype=torch.float)


def compute_metrics_from_arrays(labels, preds):
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": weighted_p,
        "recall": weighted_r,
        "f1": weighted_f1,
        "precision_macro": macro_p,
        "recall_macro": macro_r,
        "macro_F1": macro_f1,
    }


@torch.no_grad()
def evaluate_model(model, data_loader, device):
    model.eval()
    all_labels = []
    all_preds = []
    total_loss = 0.0

    for batch in data_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        outputs = model(**batch)
        total_loss += outputs["loss"].item()
        preds = outputs["logits"].argmax(dim=-1)
        all_labels.extend(batch["labels"].detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())

    labels = np.array(all_labels)
    preds = np.array(all_preds)
    metrics = compute_metrics_from_arrays(labels, preds)
    metrics["loss"] = total_loss / max(1, len(data_loader))
    return metrics, labels, preds


def train_model(model, train_loader, val_loader, run_name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        (param for param in model.parameters() if param.requires_grad),
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_state = None
    best_macro_F1 = -1.0

    # Run all epochs for a fixed experimental protocol; keep the best validation checkpoint.
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader, start=1):
            batch = {key: value.to(device) for key, value in batch.items()}
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(**batch)
                loss = outputs["loss"] / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()
            total_loss += loss.item() * GRAD_ACCUM_STEPS

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        val_metrics, _, _ = evaluate_model(model, val_loader, device)
        train_loss = total_loss / max(1, len(train_loader))
        print(
            f"{run_name} | Epoch {epoch:02d} | train_loss={train_loss:.4f} "
            f"| val_loss={val_metrics['loss']:.4f} | val_macro_F1={val_metrics['macro_F1']:.4f}"
        )

        if val_metrics["macro_F1"] > best_macro_F1:
            best_macro_F1 = val_metrics["macro_F1"]
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, device


def evaluate_and_plot(model, device, test_loader, run_name):
    metrics, labels, preds = evaluate_model(model, test_loader, device)
    print(f"{run_name} test metrics:", metrics)
    print(classification_report(
        labels,
        preds,
        target_names=label_encoder.classes_,
        zero_division=0,
    ))

    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
        cmap="Blues",
    )
    plt.title(f"Confusion Matrix - {run_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    return metrics


def summarize_seed_results(results_df, metric_columns):
    summary = results_df[metric_columns].agg(["mean", "std"]).T
    summary["mean ± std"] = summary.apply(
        lambda row: f"{row['mean']:.4f} ± {row['std']:.4f}",
        axis=1,
    )
    return summary


def run_fusion_experiments(train_frame, experiment_name, run_seed=None, run_gated=True):
    if run_seed is None:
        run_seed = RANDOM_STATE
    set_all_seeds(run_seed)
    train_loader, val_loader, test_loader = make_loaders(train_frame, seed=run_seed)
    class_weights = compute_class_weights(train_frame)
    results = []

    start_time = time.perf_counter()
    naive_model = ElectraRobertaNaiveFusion(num_labels=num_labels, class_weights=class_weights)
    naive_model = enable_memory_saving(naive_model)
    naive_model, naive_device = train_model(
        naive_model,
        train_loader,
        val_loader,
        run_name=f"ELECTRA+RoBERTa Naive Fusion ({experiment_name}, seed={run_seed})",
    )
    naive_metrics = evaluate_and_plot(
        naive_model,
        naive_device,
        test_loader,
        run_name=f"ELECTRA+RoBERTa Naive Fusion ({experiment_name}, seed={run_seed})",
    )
    naive_metrics["runtime_seconds"] = time.perf_counter() - start_time
    naive_metrics["seed"] = run_seed
    results.append({"model": f"ELECTRA+RoBERTa Naive Fusion ({experiment_name})", **naive_metrics})
    print(f"ELECTRA+RoBERTa Naive Fusion ({experiment_name}) | seed={run_seed} | runtime_seconds={naive_metrics['runtime_seconds']:.2f}")

    if run_gated:
        start_time = time.perf_counter()
        gated_model = ElectraRobertaGatedFusion(num_labels=num_labels, class_weights=class_weights)
        gated_model = enable_memory_saving(gated_model)
        gated_model, gated_device = train_model(
            gated_model,
            train_loader,
            val_loader,
            run_name=f"ELECTRA+RoBERTa Gated Fusion ({experiment_name}, seed={run_seed})",
        )
        gated_metrics = evaluate_and_plot(
            gated_model,
            gated_device,
            test_loader,
            run_name=f"ELECTRA+RoBERTa Gated Fusion ({experiment_name}, seed={run_seed})",
        )
        gated_metrics["runtime_seconds"] = time.perf_counter() - start_time
        gated_metrics["seed"] = run_seed
        results.append({"model": f"ELECTRA+RoBERTa Gated Fusion ({experiment_name})", **gated_metrics})
        print(f"ELECTRA+RoBERTa Gated Fusion ({experiment_name}) | seed={run_seed} | runtime_seconds={gated_metrics['runtime_seconds']:.2f}")

    return results

In [ ]:
# --- Shared-ELECTRA SERF model ---
def cls_pool(outputs):
    return outputs.last_hidden_state[:, 0, :]


def freeze_lower_encoder_layers(encoder):
    # Freeze lower transformer layers to reduce overfitting on short requirement texts.
    for layer in encoder.encoder.layer[:8]:
        for param in layer.parameters():
            param.requires_grad = False


def set_module_trainable(module, trainable):
    for param in module.parameters():
        param.requires_grad = trainable


def report_trainable_and_frozen_parameters(model, model_name):
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    frozen_params = sum(param.numel() for param in model.parameters() if not param.requires_grad)
    print(f"{model_name} trainable parameters: {trainable_params:,}")
    print(f"{model_name} frozen parameters: {frozen_params:,}")


class SharedElectraRobertaSERF(nn.Module):
    """SERF with one shared ELECTRA encoder and conditional RoBERTa activation."""

    def __init__(self, num_labels, class_weights=None):
        super().__init__()
        self.electra_encoder = AutoModel.from_pretrained(ELECTRA_MODEL_NAME)
        self.roberta_encoder = AutoModel.from_pretrained(ROBERTA_MODEL_NAME)
        electra_hidden = self.electra_encoder.config.hidden_size
        roberta_hidden = self.roberta_encoder.config.hidden_size
        fusion_hidden = 768

        self.selector_head = nn.Sequential(
            nn.LayerNorm(electra_hidden),
            nn.Linear(electra_hidden, electra_hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(electra_hidden, num_labels),
        )
        self.fusion_head = nn.Sequential(
            nn.LayerNorm(electra_hidden + roberta_hidden),
            nn.Linear(electra_hidden + roberta_hidden, fusion_hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(fusion_hidden, num_labels),
        )
        self.register_buffer(
            "class_weights",
            class_weights if class_weights is not None else torch.ones(num_labels),
        )
        self.training_stage = "selector"
        self.configure_selector_stage(report=False)
        report_trainable_and_frozen_parameters(self, "SharedElectraRobertaSERF")

    def enable_memory_saving(self):
        if USE_GRADIENT_CHECKPOINTING:
            for encoder in (self.electra_encoder, self.roberta_encoder):
                if hasattr(encoder, "gradient_checkpointing_enable"):
                    encoder.gradient_checkpointing_enable()
        return self

    def configure_selector_stage(self, report=True):
        set_module_trainable(self.electra_encoder, True)
        freeze_lower_encoder_layers(self.electra_encoder)
        set_module_trainable(self.selector_head, True)
        set_module_trainable(self.roberta_encoder, False)
        set_module_trainable(self.fusion_head, False)
        self.training_stage = "selector"
        if report:
            report_trainable_and_frozen_parameters(self, "SERF selector stage")

    def configure_fusion_stage(self, report=True):
        # Preserve the trained selector and reuse its fixed ELECTRA representation.
        set_module_trainable(self.electra_encoder, False)
        set_module_trainable(self.selector_head, False)
        set_module_trainable(self.roberta_encoder, True)
        freeze_lower_encoder_layers(self.roberta_encoder)
        set_module_trainable(self.fusion_head, True)
        self.training_stage = "fusion"
        if report:
            report_trainable_and_frozen_parameters(self, "SERF fusion stage")

    def configure_inference(self):
        set_module_trainable(self, False)
        self.training_stage = "inference"
        self.eval()

    def encode_electra(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        return cls_pool(self.electra_encoder(**kwargs))

    def encode_roberta(self, input_ids, attention_mask):
        return cls_pool(self.roberta_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ))

    def selector_logits_from_representation(self, h_electra):
        return self.selector_head(h_electra)

    def fusion_logits_from_representations(self, h_electra, h_roberta):
        return self.fusion_head(torch.cat([h_electra, h_roberta], dim=-1))

    def forward(
        self,
        electra_input_ids,
        electra_attention_mask,
        roberta_input_ids=None,
        roberta_attention_mask=None,
        labels=None,
        electra_token_type_ids=None,
        **kwargs,
    ):
        if self.training_stage == "selector":
            h_electra = self.encode_electra(
                electra_input_ids,
                electra_attention_mask,
                electra_token_type_ids,
            )
            logits = self.selector_logits_from_representation(h_electra)
        elif self.training_stage == "fusion":
            # train() is called by the loop, so explicitly keep the frozen selector deterministic.
            self.electra_encoder.eval()
            with torch.no_grad():
                h_electra = self.encode_electra(
                    electra_input_ids,
                    electra_attention_mask,
                    electra_token_type_ids,
                )
            h_roberta = self.encode_roberta(roberta_input_ids, roberta_attention_mask)
            logits = self.fusion_logits_from_representations(h_electra, h_roberta)
        else:
            raise RuntimeError("Use the explicit SERF inference functions after configure_inference().")

        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels, weight=self.class_weights)
        return {"loss": loss, "logits": logits}

In [ ]:
# --- Shared-ELECTRA SERF selective routing and profiling ---
RISKY_CLASSES = {"A", "L", "FT", "MN", "PO", "SC", "PE", "O", "US"}
RISKY_LABEL_IDS = {label2id[label] for label in RISKY_CLASSES if label in label2id}
TAU_VALUES = [0.70, 0.80, 0.85, 0.90]
DELTA_VALUES = [0.15, 0.20]

SERF_METRIC_COLUMNS = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "precision_macro",
    "recall_macro",
    "macro_F1",
    "loss",
    "route_to_fusion_rate",
    "electra_only_rate",
    "average_active_encoder_passes",
    "mean_confidence",
    "mean_margin",
    "mean_entropy",
    "pretokenized_latency_ms_per_sample",
    "p50_latency_ms_per_sample",
    "p95_latency_ms_per_sample",
    "throughput_samples_per_second",
    "peak_gpu_memory_mb",
    "selector_train_seconds",
    "fusion_train_seconds",
    "threshold_search_seconds",
    "routed_test_seconds",
    "runtime_seconds",
]


def collect_runtime_environment():
    environment = {
        "python_version": sys.version.split()[0],
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "processor": platform.processor() or "not explicitly reported by runtime",
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    }
    if hasattr(os, "sysconf") and "SC_PAGE_SIZE" in os.sysconf_names:
        total_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
        environment["system_ram_gb"] = total_bytes / (1024 ** 3)
    else:
        environment["system_ram_gb"] = None
    if torch.cuda.is_available():
        properties = torch.cuda.get_device_properties(0)
        environment["gpu_name"] = torch.cuda.get_device_name(0)
        environment["gpu_memory_gb"] = properties.total_memory / (1024 ** 3)
    else:
        environment["gpu_name"] = None
        environment["gpu_memory_gb"] = None
    return environment


def synchronize_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def predicted_risky_mask(preds, risky_label_ids):
    risky_mask = torch.zeros_like(preds, dtype=torch.bool)
    for risky_id in risky_label_ids:
        risky_mask |= preds.eq(int(risky_id))
    return risky_mask


@torch.no_grad()
def forward_serf_batch(model, batch, mode, tau, delta, risky_label_ids):
    if mode not in {"selector_only", "always_fusion", "selective"}:
        raise ValueError(f"Unsupported inference mode: {mode}")

    h_electra = model.encode_electra(
        batch["electra_input_ids"],
        batch["electra_attention_mask"],
        batch.get("electra_token_type_ids"),
    )
    electra_logits = model.selector_logits_from_representation(h_electra)
    electra_probs = torch.softmax(electra_logits, dim=-1)
    top2 = torch.topk(electra_probs, k=2, dim=-1).values
    confidence = top2[:, 0]
    margin = top2[:, 0] - top2[:, 1]
    entropy = -(electra_probs * torch.log(electra_probs.clamp_min(1e-12))).sum(dim=-1)
    electra_preds = electra_probs.argmax(dim=-1)

    if mode == "selector_only":
        route_to_fusion = torch.zeros_like(electra_preds, dtype=torch.bool)
    elif mode == "always_fusion":
        route_to_fusion = torch.ones_like(electra_preds, dtype=torch.bool)
    else:
        easy_mask = (
            (confidence >= tau)
            & (margin >= delta)
            & (~predicted_risky_mask(electra_preds, risky_label_ids))
        )
        route_to_fusion = ~easy_mask

    final_logits = electra_logits.clone()
    if route_to_fusion.any():
        h_roberta = model.encode_roberta(
            batch["roberta_input_ids"][route_to_fusion],
            batch["roberta_attention_mask"][route_to_fusion],
        )
        fusion_logits = model.fusion_logits_from_representations(
            h_electra[route_to_fusion],
            h_roberta,
        )
        final_logits[route_to_fusion] = fusion_logits

    return {
        "logits": final_logits,
        "route_to_fusion": route_to_fusion,
        "confidence": confidence,
        "margin": margin,
        "entropy": entropy,
    }


@torch.no_grad()
def evaluate_serf_mode(model, data_loader, device, mode, tau, delta, risky_label_ids):
    model.eval()
    all_labels = []
    all_preds = []
    all_route_to_fusion = []
    all_confidence = []
    all_margin = []
    all_entropy = []
    total_loss = 0.0

    for batch in data_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        outputs = forward_serf_batch(model, batch, mode, tau, delta, risky_label_ids)
        labels = batch["labels"]
        logits = outputs["logits"]
        preds = logits.argmax(dim=-1)
        total_loss += nn.functional.cross_entropy(
            logits,
            labels,
            weight=model.class_weights,
        ).item()
        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_route_to_fusion.extend(outputs["route_to_fusion"].detach().cpu().numpy())
        all_confidence.extend(outputs["confidence"].detach().cpu().numpy())
        all_margin.extend(outputs["margin"].detach().cpu().numpy())
        all_entropy.extend(outputs["entropy"].detach().cpu().numpy())

    labels = np.array(all_labels)
    preds = np.array(all_preds)
    route_to_fusion = np.array(all_route_to_fusion, dtype=bool)
    route_rate = float(route_to_fusion.mean())
    metrics = compute_metrics_from_arrays(labels, preds)
    metrics["loss"] = total_loss / max(1, len(data_loader))
    metrics["route_to_fusion_rate"] = route_rate
    metrics["electra_only_rate"] = 1.0 - route_rate
    metrics["average_active_encoder_passes"] = 1.0 + route_rate
    metrics["mean_confidence"] = float(np.mean(all_confidence))
    metrics["mean_margin"] = float(np.mean(all_margin))
    metrics["mean_entropy"] = float(np.mean(all_entropy))
    metrics["tau"] = tau
    metrics["delta"] = delta
    metrics["inference_mode"] = mode
    return metrics, labels, preds


def evaluate_selective_routing(model, data_loader, device, tau, delta, risky_label_ids):
    return evaluate_serf_mode(
        model,
        data_loader,
        device,
        mode="selective",
        tau=tau,
        delta=delta,
        risky_label_ids=risky_label_ids,
    )


def tune_serf_thresholds(model, val_loader, device):
    rows = []
    for tau in TAU_VALUES:
        for delta in DELTA_VALUES:
            metrics, _, _ = evaluate_selective_routing(
                model,
                val_loader,
                device,
                tau=tau,
                delta=delta,
                risky_label_ids=RISKY_LABEL_IDS,
            )
            rows.append(metrics)

    threshold_df = pd.DataFrame(rows)
    best_row = threshold_df.sort_values(
        ["macro_F1", "route_to_fusion_rate", "accuracy"],
        ascending=[False, True, False],
    ).iloc[0].to_dict()
    return threshold_df, best_row


@torch.no_grad()
def benchmark_serf_mode(model, data_loader, device, mode, tau, delta, risky_label_ids):
    model.eval()

    for _ in range(BENCHMARK_WARMUP_ROUNDS):
        for batch in data_loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            forward_serf_batch(model, batch, mode, tau, delta, risky_label_ids)
    synchronize_device(device)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    repeat_seconds = []
    for _ in range(BENCHMARK_REPEATS):
        synchronize_device(device)
        start = time.perf_counter()
        for batch in data_loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            forward_serf_batch(model, batch, mode, tau, delta, risky_label_ids)
        synchronize_device(device)
        repeat_seconds.append(time.perf_counter() - start)

    sample_count = len(data_loader.dataset)
    per_sample_ms = np.array(repeat_seconds) * 1000.0 / sample_count
    total_samples = sample_count * BENCHMARK_REPEATS
    total_seconds = float(np.sum(repeat_seconds))
    peak_memory_mb = (
        torch.cuda.max_memory_allocated(device) / (1024 ** 2)
        if device.type == "cuda"
        else np.nan
    )
    return {
        "pretokenized_latency_ms_per_sample": float(np.mean(per_sample_ms)),
        "p50_latency_ms_per_sample": float(np.percentile(per_sample_ms, 50)),
        "p95_latency_ms_per_sample": float(np.percentile(per_sample_ms, 95)),
        "repeat_latency_ms_per_sample": per_sample_ms.tolist(),
        "throughput_samples_per_second": total_samples / total_seconds,
        "peak_gpu_memory_mb": float(peak_memory_mb),
        "benchmark_repeats": BENCHMARK_REPEATS,
        "benchmark_warmup_rounds": BENCHMARK_WARMUP_ROUNDS,
        "tokenization_included": False,
        "benchmark_scope": "pretokenized test-loader inference including device transfer",
        "inference_precision": "FP32",
    }


def plot_serf_confusion_matrix(labels, preds, run_name):
    print(classification_report(
        labels,
        preds,
        target_names=label_encoder.classes_,
        zero_division=0,
    ))
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
        cmap="Blues",
    )
    plt.title(f"Confusion Matrix - {run_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


def run_serf_experiment(train_frame, experiment_name, run_seed=None):
    if run_seed is None:
        run_seed = RANDOM_STATE

    total_start = time.perf_counter()
    class_weights = compute_class_weights(train_frame)

    set_all_seeds(run_seed)
    selector_train_loader, selector_val_loader, _ = make_loaders(train_frame, seed=run_seed)
    model = SharedElectraRobertaSERF(num_labels=num_labels, class_weights=class_weights)
    model.enable_memory_saving()
    model.configure_selector_stage()
    selector_start = time.perf_counter()
    model, device = train_model(
        model,
        selector_train_loader,
        selector_val_loader,
        run_name=f"Shared ELECTRA selector ({experiment_name}, seed={run_seed})",
    )
    selector_train_seconds = time.perf_counter() - selector_start

    # Recreate seeded loaders for the second stage; data and split remain unchanged.
    set_all_seeds(run_seed)
    fusion_train_loader, fusion_val_loader, fusion_test_loader = make_loaders(train_frame, seed=run_seed)
    model.configure_fusion_stage()
    fusion_start = time.perf_counter()
    model, device = train_model(
        model,
        fusion_train_loader,
        fusion_val_loader,
        run_name=f"Shared-ELECTRA RoBERTa fusion ({experiment_name}, seed={run_seed})",
    )
    fusion_train_seconds = time.perf_counter() - fusion_start
    model.configure_inference()

    threshold_start = time.perf_counter()
    threshold_df, best_threshold = tune_serf_thresholds(model, fusion_val_loader, device)
    threshold_search_seconds = time.perf_counter() - threshold_start

    synchronize_device(device)
    test_start = time.perf_counter()
    test_metrics, test_labels, test_preds = evaluate_selective_routing(
        model,
        fusion_test_loader,
        device,
        tau=best_threshold["tau"],
        delta=best_threshold["delta"],
        risky_label_ids=RISKY_LABEL_IDS,
    )
    synchronize_device(device)
    routed_test_seconds = time.perf_counter() - test_start
    training_and_selection_seconds = time.perf_counter() - total_start

    mode_rows = []
    for mode in ("selector_only", "always_fusion", "selective"):
        mode_metrics, _, _ = evaluate_serf_mode(
            model,
            fusion_test_loader,
            device,
            mode=mode,
            tau=best_threshold["tau"],
            delta=best_threshold["delta"],
            risky_label_ids=RISKY_LABEL_IDS,
        )
        benchmark_metrics = benchmark_serf_mode(
            model,
            fusion_test_loader,
            device,
            mode=mode,
            tau=best_threshold["tau"],
            delta=best_threshold["delta"],
            risky_label_ids=RISKY_LABEL_IDS,
        )
        mode_rows.append({
            "model": "Shared-ELECTRA SERF",
            "experiment": experiment_name,
            "seed": run_seed,
            "mode": mode,
            "selected_tau": best_threshold["tau"],
            "selected_delta": best_threshold["delta"],
            **mode_metrics,
            **benchmark_metrics,
        })

    mode_results_df = pd.DataFrame(mode_rows)
    mode_lookup = {row["mode"]: row for row in mode_rows}
    selective_benchmark = mode_lookup["selective"]
    always_fusion_result = mode_lookup["always_fusion"]
    selector_only_result = mode_lookup["selector_only"]

    run_name = f"Shared-ELECTRA SERF ({experiment_name}, seed={run_seed})"
    print(
        f"{run_name} | selected_tau={best_threshold['tau']:.2f} "
        f"| selected_delta={best_threshold['delta']:.2f} "
        f"| val_macro_F1={best_threshold['macro_F1']:.4f} "
        f"| test_macro_F1={test_metrics['macro_F1']:.4f} "
        f"| route_to_fusion_rate={test_metrics['route_to_fusion_rate']:.4f}"
    )
    plot_serf_confusion_matrix(test_labels, test_preds, run_name)

    result = {
        "model": f"Shared-ELECTRA SERF ({experiment_name})",
        "seed": run_seed,
        "selected_tau": best_threshold["tau"],
        "selected_delta": best_threshold["delta"],
        "val_accuracy": best_threshold["accuracy"],
        "val_macro_F1": best_threshold["macro_F1"],
        "val_route_to_fusion_rate": best_threshold["route_to_fusion_rate"],
        **test_metrics,
        "pretokenized_latency_ms_per_sample": selective_benchmark["pretokenized_latency_ms_per_sample"],
        "p50_latency_ms_per_sample": selective_benchmark["p50_latency_ms_per_sample"],
        "p95_latency_ms_per_sample": selective_benchmark["p95_latency_ms_per_sample"],
        "throughput_samples_per_second": selective_benchmark["throughput_samples_per_second"],
        "peak_gpu_memory_mb": selective_benchmark["peak_gpu_memory_mb"],
        "always_fusion_macro_F1": always_fusion_result["macro_F1"],
        "always_fusion_latency_ms_per_sample": always_fusion_result["pretokenized_latency_ms_per_sample"],
        "selector_only_macro_F1": selector_only_result["macro_F1"],
        "selector_only_latency_ms_per_sample": selector_only_result["pretokenized_latency_ms_per_sample"],
        "parameter_count": sum(param.numel() for param in model.parameters()),
        "selector_train_seconds": selector_train_seconds,
        "fusion_train_seconds": fusion_train_seconds,
        "threshold_search_seconds": threshold_search_seconds,
        "routed_test_seconds": routed_test_seconds,
        "training_and_selection_seconds": training_and_selection_seconds,
        "runtime_seconds": time.perf_counter() - total_start,
    }

    threshold_df = threshold_df.copy()
    threshold_df["model"] = f"Shared-ELECTRA SERF ({experiment_name})"
    threshold_df["seed"] = run_seed
    threshold_df["selected"] = (
        (threshold_df["tau"] == best_threshold["tau"])
        & (threshold_df["delta"] == best_threshold["delta"])
    )
    return result, threshold_df, mode_results_df

In [ ]:
# --- SERF / CA-ERF WITH Augmentation: 3 fixed training seeds ---
# Augmentation is generated from the training split only; validation and test remain untouched.
set_all_seeds(SPLIT_SEED)
augmented_rows = []
train_counts = train_df["class"].value_counts().to_dict()

for cls, target_count in AUGMENTATION_TARGETS.items():
    current_count = train_counts.get(cls, 0)
    needed = max(0, target_count - current_count)
    if needed == 0:
        continue
    class_df = train_df[train_df["class"] == cls].reset_index(drop=True)
    if class_df.empty:
        continue
    for _ in range(needed):
        row = class_df.sample(n=1, random_state=random.randint(0, 10**9)).iloc[0]
        aug_text = augment_requirement_text(row["RequirementText"])
        augmented_rows.append({
            "RequirementText": aug_text,
            "class": cls,
            "model_text": aug_text,
            "label": int(row["label"]),
        })

df_aug = pd.DataFrame(augmented_rows)
train_aug_df = pd.concat([train_df, df_aug], ignore_index=True) if len(df_aug) > 0 else train_df.copy()
train_aug_df = train_aug_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Augmented rows added:", len(df_aug))
print("Augmented train distribution:", dict(train_aug_df["class"].value_counts()))

serf_aug_results = []
serf_aug_threshold_rows = []
serf_aug_mode_rows = []

for run_seed in TRAIN_SEEDS:
    seed_result, seed_threshold_df, seed_mode_df = run_serf_experiment(
        train_aug_df,
        "with augmentation",
        run_seed=run_seed,
    )
    serf_aug_results.append(seed_result)
    serf_aug_threshold_rows.append(seed_threshold_df)
    serf_aug_mode_rows.append(seed_mode_df)

serf_aug_results_df = pd.DataFrame(serf_aug_results)
serf_aug_threshold_results_df = pd.concat(serf_aug_threshold_rows, ignore_index=True)
serf_aug_mode_results_df = pd.concat(serf_aug_mode_rows, ignore_index=True)
serf_aug_summary = summarize_seed_results(serf_aug_results_df, SERF_METRIC_COLUMNS)

print("Shared-ELECTRA SERF with augmentation validation threshold search:")
display(serf_aug_threshold_results_df)
print("Shared-ELECTRA SERF with augmentation matched inference modes:")
display(serf_aug_mode_results_df)
print("Shared-ELECTRA SERF with augmentation seed-level test results:")
display(serf_aug_results_df)
print("Shared-ELECTRA SERF with augmentation mean ± std:")
display(serf_aug_summary)


print("Recorded runtime environment:")
display(pd.DataFrame([collect_runtime_environment()]))